# When the Noise Model Is Missing

Every file starts at `noise_type='unknown'`: until `set_noise` is called, nothing in the project says how large the scatter in the data should be. Fits run anyway, and they still quote `stderr`, profiled confidence intervals and MCMC widths — lmfit supplies the missing variance by estimating one constant from the residuals themselves.

[`12_uncertainty_mcmc`](../12_uncertainty_mcmc/example.ipynb) declares the photon-counting noise that notebook 01's data carry and reads the three tiers of uncertainty off a fit that knows it. This notebook drops the declaration and repeats those fits on the same data, so every width here has 12's beside it, computed in the same kernel.

The lesson in one line: **the three unweighted tiers agree with each other and are wrong together**, so agreement between tiers is not calibration.

Workflow:

0. **Re-run example 12** (which re-runs 01): its fits, its truth values and its Cramér-Rao bound stay live in the kernel.
1. **Drop the declaration**: what `unknown` means for the residual, the covariance, the intervals and the chain — and which columns `compare_models` withholds once a file's fits no longer share one noise model.
2. **Refit the baseline** with 12's settings: the `__lnsigma` nuisance parameter, and the χ²_red gate that is not there.
3. **The three tiers next to 12's**: the same shift in every tier, against the Cramér-Rao bound and the pulls.
4. **The 2D chain twice**: started at the derived noise scale and ten times too low, and which diagnostics fail to tell them apart.
5. **The Gaussian fallback**: one constant σ estimated from the baseline block — the right declaration where the noise is uniform, and a gate that fires when it is not.


## 0. Re-run Example Notebook 12


In [ ]:
%%capture
# %%capture swallows notebook 12's plots/output (it re-runs 01 in turn)

# %cd into notebook 12's dir so its own preamble resolves
%cd -q "../12_uncertainty_mcmc"
# %run executes that notebook in the current kernel (variables stay live)
%run example.ipynb
# back by name, not %cd -: 12's preamble used that, so the directory IPython
# calls "previous" is notebook 01's
%cd -q "../13_missing_noise_model"


In [ ]:
# Sanity check: notebook 12 should have left the file with Poisson noise
# declared and two fits of each kind in the history: 01's, then its own.
n_baseline = len(project.results.find(file=file.name, fit_type="baseline"))
n_2d = len(project.results.find(file=file.name, fit_type="2d"))
assert (file.noise_type, n_baseline, n_2d) == ("poisson", 2, 2), (
    "notebook 12 did not run to completion"
)
print(f"OK: file {file.name!r} carries noise_type={file.noise_type!r}, "
      f"{n_baseline} baseline fits and {n_2d} 2D fits")


In [ ]:
import numpy as np
import pandas as pd

from trspecfit.utils.lmfit import MC


## 1. Drop the Declaration

`set_noise("unknown")` puts the file back where every file starts. Four things follow from it:

- **The residual** is plain `data − model`, in data units. Nothing marks some points as noisier than others, so every point pulls on the fit with the same strength.
- **The covariance** is rescaled by the fit's own reduced χ²: lmfit estimates a single variance from the residuals and spreads it over the whole fit window. `stderr` comes out of that rescaled matrix.
- **The intervals** from `try_ci=1` use `lmfit.conf_interval`'s default F-test, which profiles out that same unknown variance instead of thresholding on χ² directly.
- **The chain** samples the noise scale as an extra parameter, `__lnsigma`, riding along with the model parameters.

That machinery is right when the noise really is one unknown constant. Counting noise is not: its variance is the count, so it grows with the signal, and no single constant describes both the peak and the background.

This file's baseline fits now span two noise models: notebook 01's, made before any declaration, and notebook 12's Poisson fit. `chi2`, `chi2_red`, `aic` and `bic` are each expressed in the units of the noise model that produced them. `compare_models` therefore withholds all four from its default columns for that group, and raises when one is asked for by name. What is left are the unweighted `chi2_red_raw` and `r2`, which do not depend on a noise model.


In [ ]:
# back to the default every file starts at
file.set_noise("unknown")
print(f"noise_type = {file.noise_type!r}")


In [ ]:
# the calibrated columns are gone: this file's baseline fits span two noise
# models (01's 'unknown' and 12's 'poisson')
file.compare_models(fit_type="baseline")


In [ ]:
# asking for one of them anyway
try:
    file.compare_models(fit_type="baseline", metrics=["chi2_red"])
except ValueError as err:
    print(err)


## 2. The Same Fit, Unweighted

Same model, same data, same call as 12 — reusing its `mc` object rather than a copy of its numbers, so the noise declaration is the only difference. The fit prints the two optimizer stages; then the profiled intervals, where the `GLP_01_m` profile runs into its lower limit at 0 and lmfit warns *Bound reached with prob(GLP_01_m=0)*; then the chain and its diagnostics.


In [ ]:
# 12's baseline call, unchanged: two stages, profiled CIs, and its chain settings
file.fit_baseline(model_name="base", stages=2, try_ci=1, mc_settings=mc)


In [ ]:
# what ran, from the new slot: is_weighted is False and the sigma knobs are back
slot_unknown = project.results.find(file=file.name, fit_type="baseline")[-1]
print(f"resolved MC settings: {slot_unknown.fit_settings['mc']}")
print(f"chi2_red = {slot_unknown.metrics['chi2_red']}")


In [ ]:
# tier 3 under 'unknown': one extra row for the sampled noise scale
mcmc_unknown = file.get_mcmc(fit_type="baseline")
print(f"acceptance fraction: {mcmc_unknown.acceptance_fraction.mean():.2f}")
mcmc_unknown.table


`__lnsigma` is the noise scale emcee samples when the residual is not already in units of σ: one constant σ over the fit window, its logarithm sampled alongside the model parameters. It costs a dimension, so `nwalkers` derives as 2 × (free parameters + 1); that is still under the floor of 20 here, and the chain runs the same 20 walkers as 12's. It also needs a starting point. `sigma_ini` defaults to the fit's RMS residual, with the sampling bounds two decades either side of it, and §4 starts it somewhere else on purpose.

In the corner plot the fit drew, the `__lnsigma` row barely correlates with any model parameter — none of its correlations reach the 0.1 threshold lmfit's fit report prints. The extra dimension is cheap, and it is not where the trouble is. The posterior above is a fair posterior for the model *one constant, unknown σ*; that model is the problem, on data whose σ changes from pixel to pixel.

Nothing in the output says so. `chi2_red` is the weighted χ² per degree of freedom, and under `unknown` there is no weighted χ², so the slot records `nan`: the gate 12 reads after every fit does not exist here. A misweighted fit and a well-weighted one print the same diagnostics.


## 3. The Three Tiers Next to 12's

Notebook 12's baseline fit is still in the history. Pick it out by the noise it was made under, pass its handle to the same accessors, and the two sets of tiers line up parameter by parameter.


In [ ]:
# 12's baseline slot, identified by its noise model rather than its position
slot_poisson = next(
    s
    for s in project.results.find(file=file.name, fit_type="baseline")
    if s.noise_type == "poisson"
)
# the three tiers of each fit, keyed by parameter name
pars_poisson = file.get_parameters(handle=slot_poisson.handle).set_index("name")
pars_unknown = file.get_parameters(handle=slot_unknown.handle).set_index("name")
tiers = {
    ("stderr", "poisson"): pars_poisson["stderr"],
    ("stderr", "unknown"): pars_unknown["stderr"],
    ("ci", "poisson"): one_sigma_halfwidth(
        file.get_confidence_intervals(handle=slot_poisson.handle)
    ),
    ("ci", "unknown"): one_sigma_halfwidth(
        file.get_confidence_intervals(handle=slot_unknown.handle)
    ),
    ("mcmc", "poisson"): one_sigma_halfwidth(
        file.get_mcmc(handle=slot_poisson.handle).table
    ),
    ("mcmc", "unknown"): one_sigma_halfwidth(mcmc_unknown.table),
}
# the free model parameters, in the order the fit reports them
free_pars = [n for n, row in pars_unknown.iterrows() if row["vary"] and n in truth]


In [ ]:
# one row per parameter and tier: the two half-widths and their ratio
rows_tiers = [
    {
        "parameter": name,
        "tier": tier,
        "poisson": tiers[tier, "poisson"][name],
        "unknown": tiers[tier, "unknown"][name],
        "unknown / poisson": tiers[tier, "unknown"][name]
        / tiers[tier, "poisson"][name],
    }
    for name in free_pars
    for tier in ("stderr", "ci", "mcmc")
]
pd.DataFrame(rows_tiers).round(4)


In [ ]:
# which set of error bars is the right size: the Cramér-Rao bound notebook 12
# computed from the truth model, and the pull each fit gets against the truth
rows_cal = [
    {
        "parameter": name,
        "truth": truth[name],
        "fit_unknown": pars_unknown.loc[name, "value"],
        "crb": bound[name],
        "stderr_poisson / crb": pars_poisson.loc[name, "stderr"] / bound[name],
        "stderr_unknown / crb": pars_unknown.loc[name, "stderr"] / bound[name],
        "pull_poisson": (pars_poisson.loc[name, "value"] - truth[name])
        / pars_poisson.loc[name, "stderr"],
        "pull_unknown": (pars_unknown.loc[name, "value"] - truth[name])
        / pars_unknown.loc[name, "stderr"],
    }
    for name in free_pars
]
pd.DataFrame(rows_cal).round(4)


**The tiers move together.** For each parameter the ratio barely changes from one tier to the next. The three are not independent estimates that happen to agree — all three are built on the single variance lmfit read off the residuals, and they inherit its error together. Agreement between tiers checks the shape of the χ² surface, not the noise model.

**The direction depends on where the counts are.** The two background parameters come out wider than under the declared noise, the four peak parameters narrower. One variance spread over the whole window overstates the noise on the flat background, where the counts and their Poisson scatter are small, and understates it under the peak, where the scatter is largest. The fit weights every pixel alike, and the error bars follow.

**The bound settles which set is right.** Under the declared noise `stderr` sits on the Cramér-Rao bound; under `unknown` all four peak parameters land below it, `GLP_01_A` at little more than half. A standard deviation below the bound is not something an unbiased estimator can reach, so those error bars are not standard deviations of anything. (`GLP_01_F` and `GLP_01_m` carry a wrinkle of their own, which is why even the declared-noise column is not exactly 1 for them: the bound is computed from the truth model and the covariance at the fitted point.)

The pulls do not catch any of this. Each uses its own fit's value and error bar; the two objectives have slightly different minima. Under both declarations the pulls stay unremarkable, the largest a little over one σ. Six numbers from one realization cannot resolve a factor of two in a width. That is the uncomfortable part of the default: every check available from inside the fit looks healthy.


## 4. The 2D Chain, Twice

Notebook 12 ran one short chain on the 2D fit to show how to read its convergence. Here the same short chain runs twice under `unknown`, differing in one input: where the sampled noise scale starts. The first takes the default, the fit's own RMS residual. The second starts ten times too low, which is what a stale or mistyped noise estimate does.

Both chains need a scale to be checked against, and the baseline block supplies one without a model. Nothing evolves in the slices notebook 01 averaged into the ground-state spectrum, so the spread over time there is noise. Where σ varies from pixel to pixel, the root-mean-square over the fitted pixels is the summary that keeps a χ²-style check honest, since χ² averages squared residuals and their expectation is σᵢ².


In [ ]:
# the model-free estimate: base_t_ind holds the slice range notebook 01 passed
# to define_baseline, e_lim the energy window it fits.
t_ind, e_ind = file.base_t_ind, file.e_lim
base_block = file.data[t_ind[0] : t_ind[1], e_ind[0] : e_ind[1]]
# one scalar for a signal-dependent sigma: the RMS over the fitted pixels
sigma_est = np.sqrt(base_block.var(axis=0, ddof=1).mean())
print(f"sigma_est = {sigma_est:.4f}  (per time slice, RMS over the fit window)")


In [ ]:
# 12's short-chain settings, reused: the start of the noise scale is derived
file.fit_2d(model_name="2D", stages=2, try_ci=0, mc_settings=mc_2d)


In [ ]:
# the same chain, started ten times too low. Everything else is read off
# 12's object rather than retyped.
mc_off = MC(
    use_mc=mc_2d.use_mc,
    steps=mc_2d.steps,
    burn=mc_2d.burn,
    seed=mc_2d.seed,
    sigma_ini=sigma_est / 10,  # the one difference
)
file.fit_2d(model_name="2D", stages=2, try_ci=0, mc_settings=mc_off)


In [ ]:
# the two chains, in history order
slots_2d = project.results.find(file=file.name, fit_type="2d")
mcmc_derived = file.get_mcmc(handle=slots_2d[-2].handle)
mcmc_off = file.get_mcmc(handle=slots_2d[-1].handle)
for label, chain, slot in (
    ("derived start", mcmc_derived, slots_2d[-2]),
    ("off-scale start", mcmc_off, slots_2d[-1]),
):
    print(
        f"{label:>16}:  sigma_ini {slot.fit_settings['mc']['sigma_ini']:.4f}"
        f"  ->  sampled sigma {np.exp(chain.lnsigma):.4f}"
        f"   acceptance {chain.acceptance_fraction.mean():.2f}"
    )
print(f"{'model-free':>16}:  sigma_est {sigma_est:.4f}")


In [ ]:
# tier 1 is the cross-check: posterior half-widths against stderr, and against
# the widths 12's weighted chain reported for the same parameters. Both chains
# sit on the same unweighted optimizer result, so one stderr column serves both.
stderr_2d = file.get_parameters(handle=slots_2d[-2].handle).set_index("name")["stderr"]
w_derived = one_sigma_halfwidth(mcmc_derived.table)
w_off = one_sigma_halfwidth(mcmc_off.table)
w_poisson = one_sigma_halfwidth(mcmc_2d.table)
rows_2d = [
    {
        "parameter": name,
        "stderr": stderr_2d[name],
        "mcmc": w_derived[name],
        "mcmc_off": w_off[name],
        "mcmc / stderr": w_derived[name] / stderr_2d[name],
        "mcmc_off / stderr": w_off[name] / stderr_2d[name],
        "mcmc_12": w_poisson[name],
    }
    for name in w_poisson
]
pd.DataFrame(rows_2d).round(4)


Both chains printed the same warning, *"The chain is shorter than 50 times the integrated autocorrelation time"*. It says emcee cannot vouch for its own τ estimate at 500 steps, not that the walkers failed to arrive. Both acceptance fractions land in or just above emcee's healthy 0.2–0.5 band. Neither diagnostic separates the two runs.

Everything that does separate them comes from outside the chain. The chain started at the derived scale sampled a noise scale that matches the model-free `sigma_est`, and its posterior widths reproduce `stderr` parameter by parameter: a short chain that arrived. The chain started ten times too low spent its 500 steps travelling toward the right scale, never reached it, and returned widths hundreds of times `stderr` — no information about the parameters at all.

What the arrived chain still gets wrong is in the last column. Its widths sit well below the ones 12's weighted chain reported, by roughly the factor §3 found for the peak parameters — every parameter free in this fit describes the peak's position.


## 5. The Gaussian Fallback

`unknown` is not the only fallback. Where the noise really is uniform Gaussian (an analog detector, dark-subtracted or difference data), a constant σ is the correct declaration rather than an approximation, and [`10_model_comparison`](../10_model_comparison/example.ipynb) fits under exactly that. The σ does not have to come from the instrument either: the baseline-block estimate above is model-free.

On counting data it is the wrong declaration, but not the same kind of wrong as no declaration at all.


In [ ]:
# one constant sigma for the whole window, from the baseline block above.
# try_ci=0 and no chain: tier 1 is enough to see what the declaration does.
file.set_noise("gaussian", sigma=sigma_est)
file.fit_baseline(model_name="base", stages=2, try_ci=0)
slot_gauss = project.results.find(file=file.name, fit_type="baseline")[-1]
print(f"chi2_red = {slot_gauss.metrics['chi2_red']:.2f}")


In [ ]:
# where a constant sigma puts the error bars, against the other two declarations
stderr_gauss = file.get_parameters(handle=slot_gauss.handle).set_index("name")["stderr"]
pd.DataFrame(
    {
        "poisson": pars_poisson["stderr"],
        "unknown": pars_unknown["stderr"],
        "gaussian": stderr_gauss,
    }
).loc[free_pars].rename_axis("parameter").round(4)


In [ ]:
# the same declaration with sigma half its estimated size
file.set_noise("gaussian", sigma=sigma_est / 2)
file.fit_baseline(model_name="base", stages=2, try_ci=0)
slot_half = project.results.find(file=file.name, fit_type="baseline")[-1]
print(f"chi2_red = {slot_half.metrics['chi2_red']:.2f}")


The `unknown` and `gaussian` columns are within a couple of percent of each other, and that is the point: declaring one constant σ weights the fit the way `unknown` already did, with the number supplied instead of estimated. The Poisson column is the one sitting on the bound (§3), and neither of the others moved toward it.

What the declaration buys is the gate. `chi2_red` is defined again: it reads ≈ 1 for the baseline-block estimate, and about four times that for a σ declared half as large. The metric goes as 1/σ², so a factor of two in the declaration is a factor of four in the gate. A constant σ that is wrong overall cannot hide.

A constant σ that is wrong *pixel by pixel* can. The gate above reads 1 with the counting noise misdescribed, because the estimate matches the residuals taken as a whole. That is the failure the bound caught in §3 and the gate cannot.


## Tips for a Missing Noise Model

**What `unknown` does**
- It is the default: a file carries it until `set_noise` is called. Fits under it minimize the unweighted residual and quote error bars from a covariance rescaled by the fit's own reduced χ².
- All three tiers stay self-consistent under it, because all three rest on that one estimated variance. Tiers that agree tell you nothing about whether the widths are right.
- There is no χ²_red gate — the metric is `nan` — and `compare_models` withholds every noise-scaled column from a group whose fits span more than one noise model.

**Recognizing the damage**
- Misweighting is structured, not a global factor: on counting data the error bars come out too small where the signal is strongest and too large on the background.
- Check from outside the fit. `sensitivity.crb` needs a model and a photon budget, no data and no fit, and an `stderr` below that bound cannot be the standard deviation of an unbiased estimator.
- Pulls against a known truth are a weak test on one dataset: a factor of two in the widths hides inside the normal scatter of a handful of parameters.

**MCMC under `unknown`**
- The chain samples the noise scale as `__lnsigma`, so `nwalkers` derives as 2 × (free parameters + 1) with a floor of 20, and `sigma_ini` has to start near the true scale. Its default, the fit's RMS residual, is where it belongs; override it only when a structured misfit has inflated that residual.
- Started far off, a short chain returns confidently wrong widths while its acceptance fraction and its 50τ warning look exactly like a healthy chain's. Compare the posterior widths against `stderr`, and the sampled scale against a model-free estimate.

**The fallback**
- `set_noise('gaussian', sigma=...)` with σ measured on a pre-trigger block is the right declaration when the noise is uniform, and restores the gate in every case. It does not fix signal-dependent noise: its error bars are the unweighted ones.

## Next Steps

- The same data and fits with the noise declared, and all three tiers against the bound: [`12_uncertainty_mcmc`](../12_uncertainty_mcmc/example.ipynb)
- A known Gaussian σ where it is the right model, and the metrics it calibrates: [`10_model_comparison`](../10_model_comparison/example.ipynb)
- The underlying fit workflow: [`01_basic_fitting`](../01_basic_fitting/example.ipynb)
